# RGB2Point — Web Images → Reviewed RGB → 1,024 / 8,192 Point Clouds

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Md-Mosabbir/localpoint3d/blob/main/notebooks/%7BUPDATED%7DRGB2Point_Google_Review_1024_8192.ipynb)

This Colab notebook uses the released, frozen RGB2Point checkpoint. It does **not** train or fine-tune RGB2Point.

## Workflow — you control when conversion starts

1. Install dependencies, clone RGB2Point, and load the released checkpoint.
2. (Optional first-run test) Upload one image and inspect its 1,024-point prediction.
3. Search **Bing Images and DuckDuckGo Images** (and/or use your own images) and collect up to 100 candidate RGB images.
4. Display contact sheets so you can inspect the images. Reject incorrect images by their displayed numbers, then download the RGB-image ZIP.
5. The notebook pauses. You must explicitly type `APPROVE` in the approval cell before any batch point-cloud conversion runs.
6. Generate and download the original **1,024-point** XYZ point clouds.
7. Interpolate the 1,024 points to exactly **8,192 points** and download that separate output.

## Important limitations

- The released checkpoint predicts 1,024 XYZ coordinates. The 8,192-point version below is a **post-processing interpolation**, not an 8,192-point prediction from RGB2Point and not newly recovered geometric detail.
- The checkpoint's stated training domain is ShapeNet chairs, airplanes, and cars. Predictions for objects such as Bangladeshi wooden benches are out-of-domain estimates, not ground-truth scans.
- RGB2Point predicts XYZ only: no point colors, metric scale, material, or reliable hidden geometry.
- PointLLM may require colored point clouds and text annotations; these XYZ files alone are not a complete PointLLM dataset. MetaCLIP is image-text, so use the reviewed RGB images with captions rather than feeding it PLY files.
- Images found by a web search are not automatically licensed for reuse. Check image permissions before training or redistribution.

**Why not Google Images?** Google now shows image results only to real browsers and blocks scripts (the old `GoogleImageCrawler` step returned 0 images, checked 2026-11-16). We do not try to get around that block. Bing and DuckDuckGo still answer scripts, and images you saved yourself (from Google or anywhere else) can be added with `OWN_IMAGES_DIR` / `UPLOAD_OWN_IMAGES`.

Any search can be rate-limited or change. That is why there are several sources: if one fails, the notebook moves on to the next. It is not guaranteed to reach 100 images for a rare object.

## 1. Install dependencies and prepare RGB2Point

In Colab, select **Runtime → Change runtime type → T4 GPU** (or another available GPU) before running these cells. The code will fall back to CPU, but generating 100 clouds will be much slower.

In [ ]:
%pip install "timm==1.0.15" "gdown>=5.2.0" "icrawler>=0.6.10" "ddgs>=9.0.0" "plotly>=5.18.0" pandas tqdm requests pillow scipy

In [ ]:
import os
import re
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/JaeLee18/RGB2point.git"
REPO_DIR = Path("/content/RGB2point")
CHECKPOINT_ID = "1Z5luy_833YV6NGiKjGhfsfEUyaQkgua1"
CHECKPOINT_PATH = Path("/content/pc1024_three.pth")

# Clone the official implementation.
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

# The checkpoint contains the complete model state_dict, including the ViT weights.
# Disable timm's separate ImageNet-weight download during model construction; then
# load the checkpoint's own ViT weights below.
model_file = REPO_DIR / "model.py"
model_source = model_file.read_text()
model_source, replacements = re.subn(
    r'("vit_base_patch16_224"\s*,\s*pretrained\s*=\s*)True',
    r'\1False',
    model_source,
    count=1,
)
if replacements != 1:
    raise RuntimeError("Could not safely patch timm pretrained=True in model.py; inspect the upstream repository version.")
model_file.write_text(model_source)

# Download the exact checkpoint linked in the authors' README.
if not CHECKPOINT_PATH.exists() or CHECKPOINT_PATH.stat().st_size < 1_000_000:
    import gdown
    result = gdown.download(id=CHECKPOINT_ID, output=str(CHECKPOINT_PATH), quiet=False)
    if not result or not CHECKPOINT_PATH.exists() or CHECKPOINT_PATH.stat().st_size < 1_000_000:
        raise RuntimeError(
            "Checkpoint download failed or returned an unexpectedly small file. "
            "Open the RGB2Point README and verify that the Google Drive checkpoint is still accessible."
        )

print("Repository:", REPO_DIR)
print("Checkpoint:", CHECKPOINT_PATH, f"({CHECKPOINT_PATH.stat().st_size / (1024**2):.1f} MiB)")

## 2. Load the released checkpoint

In [ ]:
import torch
import numpy as np
from PIL import Image, ImageOps
from IPython.display import display, Image as IPyImage

sys.path.insert(0, str(REPO_DIR))
from model import PointCloudNet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected. In Colab, enable a GPU runtime for practical batch processing.")

# Match the authors' inference.py architecture exactly.
model = PointCloudNet(
    num_views=1,
    point_cloud_size=1024,
    num_heads=4,
    dim_feedforward=2048,
)

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
except TypeError:  # Compatibility with older PyTorch releases.
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
# Be tolerant of a DataParallel prefix, while still loading strictly afterwards.
if isinstance(state_dict, dict) and state_dict and all(k.startswith("module.") for k in state_dict):
    state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

model.load_state_dict(state_dict, strict=True)
model.to(DEVICE)
model.eval()
print("RGB2Point checkpoint loaded successfully.")

## 3. Shared inference, export, and visualization helpers

These helpers are used by both the one-image smoke test and the 100-image dataset pipeline.

In [ ]:
import numpy as np
import torch
import plotly.graph_objects as go

_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)


def preprocess_rgb_image(image_path):
    """Replicate the repository's Resize(224), ToTensor, ImageNet normalization."""
    with Image.open(image_path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im = im.resize((224, 224), resample=Image.Resampling.BILINEAR)
        arr = np.asarray(im, dtype=np.float32) / 255.0
    arr = (arr.transpose(2, 0, 1) - _MEAN) / _STD
    tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)  # [1, 1, 3, 224, 224]
    return tensor.to(DEVICE, dtype=torch.float32)


@torch.inference_mode()
def predict_points(image_path):
    """Return the model's 1,024 predicted XYZ coordinates as float32 NumPy."""
    model_input = preprocess_rgb_image(image_path)
    prediction = model(model_input)
    points = prediction[0].detach().cpu().numpy().astype(np.float32)
    if points.shape != (1024, 3):
        raise ValueError(f"Unexpected model output shape {points.shape}; expected (1024, 3).")
    if not np.isfinite(points).all():
        raise ValueError("Model output contains NaN or infinite coordinates.")
    return points



def upsample_points_interpolation(points, target_points=8192, neighbors=5, seed=42):
    """
    Increase point count by interpolating between nearby existing XYZ points.

    IMPORTANT: this creates intermediate coordinates; it does not infer new
    surface detail or make the model's reconstruction more accurate.
    """
    from scipy.spatial import cKDTree

    points = np.asarray(points, dtype=np.float32)
    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError("Expected points with shape (N, 3).")
    if len(points) == 0:
        raise ValueError("Cannot upsample an empty point cloud.")
    if target_points < 1:
        raise ValueError("target_points must be at least 1.")
    if len(points) >= target_points:
        return points[:target_points].copy()
    if len(points) == 1:
        return np.repeat(points, target_points, axis=0)

    rng = np.random.default_rng(seed)
    k = min(max(2, neighbors), len(points))
    _, nearest = cKDTree(points).query(points, k=k)

    extra_count = target_points - len(points)
    anchors = rng.integers(0, len(points), size=extra_count)
    neighbor_columns = rng.integers(1, k, size=extra_count)
    neighbor_ids = nearest[anchors, neighbor_columns]
    alpha = rng.uniform(0.15, 0.85, size=(extra_count, 1)).astype(np.float32)

    new_points = (1.0 - alpha) * points[anchors] + alpha * points[neighbor_ids]
    result = np.vstack([points, new_points]).astype(np.float32)

    if result.shape != (target_points, 3):
        raise RuntimeError(f"Upsampling failed: got {result.shape}, expected ({target_points}, 3).")
    return result


def save_ply_xyz(points, output_path):
    """Write a standards-compatible ASCII PLY containing XYZ vertices only."""
    points = np.asarray(points, dtype=np.float32)
    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError("points must have shape (N, 3)")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        f.write("ply\nformat ascii 1.0\n")
        f.write(f"element vertex {len(points)}\n")
        f.write("property float x\nproperty float y\nproperty float z\n")
        f.write("end_header\n")
        np.savetxt(f, points, fmt="%.7f %.7f %.7f")
    return output_path


def show_point_cloud(points, title="RGB2Point prediction"):
    points = np.asarray(points)
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        mode="markers", marker=dict(size=2.2, opacity=0.85),
    )])
    fig.update_layout(
        title=title,
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
        margin=dict(l=0, r=0, t=45, b=0),
        height=620,
    )
    fig.show()


def point_cloud_stats(points):
    points = np.asarray(points)
    lo, hi = points.min(axis=0), points.max(axis=0)
    extent = hi - lo
    return {
        "points": int(len(points)),
        "x_extent": float(extent[0]),
        "y_extent": float(extent[1]),
        "z_extent": float(extent[2]),
        "min_coordinate": float(points.min()),
        "max_coordinate": float(points.max()),
    }

print("Inference helpers are ready.")

# FIRST-RUN ONLY: one-image smoke test

**Run these two test cells once before building a dataset.** Upload a representative object image (ideally one of the local Bangladeshi objects you care about), then inspect the point cloud from multiple angles. This does not calculate an accuracy score because a single internet image has no ground-truth 3D scan. It only lets you check whether the prediction looks structurally plausible.

After you are done evaluating, you can delete **the next two code cells**. Keep the setup and helper cells above, and keep the bulk pipeline cells below.

In [ ]:
# TEST CELL 1/2 — upload exactly one RGB image
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No image was uploaded.")

smoke_test_filename = next(iter(uploaded))
SMOKE_TEST_IMAGE = f"/content/{smoke_test_filename}"
print("Smoke-test image:", SMOKE_TEST_IMAGE)
display(IPyImage(filename=SMOKE_TEST_IMAGE))

smoke_test_points = predict_points(SMOKE_TEST_IMAGE)
SMOKE_TEST_PLY = "/content/rgb2point_smoke_test.ply"
save_ply_xyz(smoke_test_points, SMOKE_TEST_PLY)
print("Point-cloud statistics:", point_cloud_stats(smoke_test_points))
print("Saved:", SMOKE_TEST_PLY)

In [ ]:
# TEST CELL 2/2 — inspect the generated point cloud interactively
show_point_cloud(smoke_test_points, title=f"Smoke test: {smoke_test_filename}")

# Optional: download this single PLY for inspection in CloudCompare / MeshLab.
# from google.colab import files
# files.download(SMOKE_TEST_PLY)

## 4. Dataset settings — change only the object name for a new class

Images come from the sources in `IMAGE_SOURCES`, tried in that order until `TARGET_IMAGES` is reached. With `ADD_BANGLADESH_TERM=True` the search first asks for "<object> Bangladesh" and then tops up with the plain object name. This does not guarantee that every image shows a Bangladeshi-made or locally distinctive object, so the review step below still matters.

**Your own images** (optional): pictures you saved by hand from Google Images, or took yourself, are used first.
- `OWN_IMAGES_DIR`: a folder that already exists in Colab or on your mounted Drive, or
- `UPLOAD_OWN_IMAGES = True`: the next cell asks you to upload image files or a ZIP.

Set `IMAGE_SOURCES = []` to use only your own images.

> 🧠 A Python **list** keeps its order, so `["bing", "duckduckgo"]` means "try Bing first".

In [ ]:
# EDIT THIS for each dataset you want to generate.
OBJECT_QUERY = "Wooden Bench"

TARGET_IMAGES = 100
IMAGE_SOURCES = ["bing", "duckduckgo"]   # tried in this order; choose from "bing", "duckduckgo"
OWN_IMAGES_DIR = ""                      # optional: folder with images you collected yourself (used first)
UPLOAD_OWN_IMAGES = False                # True = the next cell asks you to upload images or a ZIP
ADD_BANGLADESH_TERM = True
MIN_IMAGE_SIDE = 128
MAX_IMAGE_MB = 15
REQUEST_TIMEOUT = 25
UPSAMPLED_POINT_COUNT = 8192

OUTPUT_ROOT = Path("/content/rgb2point_datasets")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

assert OBJECT_QUERY.strip(), "Please set OBJECT_QUERY to a non-empty object name."
print(f"Will collect up to {TARGET_IMAGES} images for {OBJECT_QUERY!r} from: {IMAGE_SOURCES}")
print("Output root:", OUTPUT_ROOT)

## 4b. (Optional) Upload your own images

This cell only does something when `UPLOAD_OWN_IMAGES = True`. Choose image files or one ZIP; they are put in a folder and `OWN_IMAGES_DIR` is pointed at it. Safe to run when the setting is `False`.

In [ ]:
if UPLOAD_OWN_IMAGES:
    import zipfile
    from google.colab import files

    own_dir = Path("/content/own_images") / OBJECT_QUERY.strip().lower().replace(" ", "_")
    own_dir.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        target = own_dir / Path(name).name
        target.write_bytes(data)
        if target.suffix.lower() == ".zip":       # unpack a ZIP, then remove the ZIP itself
            with zipfile.ZipFile(target) as archive:
                archive.extractall(own_dir)
            target.unlink()
    OWN_IMAGES_DIR = str(own_dir)
    print("Own images folder:", OWN_IMAGES_DIR, "-", len([p for p in own_dir.rglob("*") if p.is_file()]), "files")
else:
    print("UPLOAD_OWN_IMAGES is False: nothing to upload.")

## 5. Collect candidate RGB images (Bing, DuckDuckGo, your own)

This cell collects up to 100 valid, distinct images. It does **not** run RGB2Point. You will inspect the images in the next cell and explicitly approve them before conversion.

What it does, in order:
1. Takes your own images first, if you gave any.
2. Asks each source in `IMAGE_SOURCES` for "<object> Bangladesh", then for the plain object name, until there are enough.
3. Checks every file really is an image, is big enough, and is not a copy of one we already have (same file, or a near-identical picture).
4. Remembers where each image came from (`image_source`, `source_url`) for the manifest.

If a source fails, the error is printed and the next source is tried. Lines like `ERROR - downloader - Response status code 403` are normal: some websites refuse downloads, and those images are simply skipped.

> 🧠 `try: ... except Exception:` runs the code and, if it breaks, jumps to the `except` part instead of stopping the notebook. That is how one blocked source does not stop the others.

In [ ]:
import hashlib
import io
import time
import shutil
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from icrawler import ImageDownloader
from icrawler.builtin import BingImageCrawler
from tqdm.auto import tqdm
from PIL import Image, ImageOps

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}
# An honest name for our downloader, so websites can see who is asking.
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; RGB2PointDatasetResearch/1.0)"}


def slugify(text):
    slug = re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")
    return slug or "object"


def dhash_64(pil_image):
    small = np.asarray(pil_image.convert("L").resize((9, 8), Image.Resampling.LANCZOS))
    bits = small[:, 1:] > small[:, :-1]
    value = 0
    for bit in bits.flatten():
        value = (value << 1) | int(bit)
    return value


def hamming_distance(a, b):
    return (int(a) ^ int(b)).bit_count()


def normalize_and_validate_image(source_path, output_path):
    with Image.open(source_path) as im_check:
        im_check.verify()
    with Image.open(source_path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        width, height = im.size
        if min(width, height) < MIN_IMAGE_SIDE:
            raise ValueError(f"Image too small ({width}x{height})")
        phash = dhash_64(im)
        im.save(output_path, format="JPEG", quality=94, optimize=True)
    return {
        "original_width": int(width),
        "original_height": int(height),
        "sha256": hashlib.sha256(Path(output_path).read_bytes()).hexdigest(),
        "dhash": phash,
    }


# ---------------------------------------------------------------- image sources
# Every search function returns a list of candidates:
#   {"file": downloaded file, "image_url": where it came from, "source_page": web page (if known)}

class RememberUrlDownloader(ImageDownloader):
    """icrawler's downloader, plus a note of which URL each saved file came from."""

    def process_meta(self, task):
        if task.get("success"):
            self.url_of_file[task["filename"]] = task["file_url"]


def search_bing(query, max_num, raw_dir):
    crawler = BingImageCrawler(
        downloader_cls=RememberUrlDownloader,
        storage={"root_dir": str(raw_dir)},
        downloader_threads=4,
    )
    crawler.downloader.url_of_file = {}
    crawler.crawl(keyword=query, filters={"type": "photo"}, max_num=min(max_num, 1000),
                  min_size=(MIN_IMAGE_SIDE, MIN_IMAGE_SIDE))
    return [{"file": raw_dir / name, "image_url": url, "source_page": ""}
            for name, url in crawler.downloader.url_of_file.items()]


def download_one(job):
    """Download one image URL to a file. Returns True if it worked."""
    url, path = job
    try:
        response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        if not 2_000 < len(response.content) < MAX_IMAGE_MB * 1024 * 1024:
            return False
        path.write_bytes(response.content)
        return True
    except Exception:
        return False


def search_duckduckgo(query, max_num, raw_dir):
    from ddgs import DDGS
    results = list(DDGS().images(query, safesearch="moderate", max_results=max_num))
    jobs = [(r["image"], raw_dir / f"{i:06d}.img") for i, r in enumerate(results, start=1) if r.get("image")]
    with ThreadPoolExecutor(max_workers=8) as pool:    # 8 downloads at the same time
        worked = list(pool.map(download_one, jobs))
    page_of = {r.get("image"): r.get("url", "") for r in results}
    return [{"file": path, "image_url": url, "source_page": page_of.get(url, "")}
            for (url, path), ok in zip(jobs, worked) if ok]


def search_own_images(query, max_num, raw_dir):
    """Images you collected yourself (OWN_IMAGES_DIR). Nothing is downloaded."""
    folder = Path(OWN_IMAGES_DIR)
    if not folder.is_dir():
        raise FileNotFoundError(f"OWN_IMAGES_DIR does not exist: {folder}")
    files_found = sorted(p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES)
    return [{"file": p, "image_url": "", "source_page": ""} for p in files_found]


SEARCHERS = {"own": search_own_images, "bing": search_bing, "duckduckgo": search_duckduckgo}

# ---------------------------------------------------------------- collect
DATASET_STAMP = datetime.utcnow().strftime("%Y%m%d_%H%M%S_utc")
DATASET_DIR = OUTPUT_ROOT / f"{slugify(OBJECT_QUERY)}_{DATASET_STAMP}"
IMAGES_DIR = DATASET_DIR / "images"
POINTS_1024_DIR = DATASET_DIR / "pointclouds_1024"
POINTS_8192_DIR = DATASET_DIR / "pointclouds_8192"
RAW_DIR = DATASET_DIR / "search_raw"
for folder in [IMAGES_DIR, POINTS_1024_DIR, POINTS_8192_DIR, RAW_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# The Bangladesh query goes first; the plain query fills up what is still missing.
search_queries = [OBJECT_QUERY.strip()]
if ADD_BANGLADESH_TERM:
    search_queries.insert(0, OBJECT_QUERY.strip() + " Bangladesh")
search_query = search_queries[0]

sources = [s for s in IMAGE_SOURCES if s in SEARCHERS]
unknown = [s for s in IMAGE_SOURCES if s not in SEARCHERS]
if unknown:
    print("Ignoring unknown IMAGE_SOURCES:", unknown, "- choose from", list(SEARCHERS))
if OWN_IMAGES_DIR:
    sources.insert(0, "own")           # your own images are always used first

accepted = []
seen_sha256 = set()
seen_hashes = []
seen_urls = set()
failures = []
found_per_source = {}

for source in sources:
    for query in (["(your own images)"] if source == "own" else search_queries):
        if len(accepted) >= TARGET_IMAGES:
            break
        raw_dir = RAW_DIR / f"{source}_{slugify(query)}"
        raw_dir.mkdir(parents=True, exist_ok=True)
        print(f"\n[{source}] searching: {query!r}")
        try:
            candidates = SEARCHERS[source](query, TARGET_IMAGES * 2, raw_dir)
        except Exception as exc:
            # One source failing (blocked, rate-limited, changed) must not stop the notebook.
            print(f"[{source}] failed: {exc!r}. Moving on to the next source.")
            failures.append({"file": f"{source}: {query}", "error": repr(exc)})
            continue
        print(f"[{source}] files downloaded: {len(candidates)}")

        kept_before = len(accepted)
        for candidate in tqdm(candidates, desc="Validating and deduplicating"):
            if len(accepted) >= TARGET_IMAGES:
                break
            if candidate["image_url"] and candidate["image_url"] in seen_urls:
                continue
            image_number = len(accepted) + 1
            image_path = IMAGES_DIR / f"{slugify(OBJECT_QUERY)}_{image_number:04d}.jpg"
            try:
                details = normalize_and_validate_image(candidate["file"], image_path)
                is_copy = details["sha256"] in seen_sha256 or any(
                    hamming_distance(details["dhash"], old_hash) <= 2 for old_hash in seen_hashes)
                if is_copy:
                    image_path.unlink(missing_ok=True)
                    continue
                seen_sha256.add(details["sha256"])
                seen_hashes.append(details["dhash"])
                seen_urls.add(candidate["image_url"])
                accepted.append({
                    "sample_id": f"{slugify(OBJECT_QUERY)}_{image_number:04d}",
                    "object_label": OBJECT_QUERY,
                    "image_path": str(image_path),
                    "search_query": "" if source == "own" else query,
                    "image_source": source,
                    "source_url": candidate["image_url"],
                    "source_page": candidate["source_page"],
                    "original_width": details["original_width"],
                    "original_height": details["original_height"],
                    "sha256": details["sha256"],
                    "license_status": "NOT_VERIFIED_CHECK_BEFORE_TRAINING",
                    "manual_review_status": "PENDING",
                    "manual_review_notes": "",
                })
            except Exception as exc:
                failures.append({"file": str(candidate["file"]), "error": str(exc)})
                image_path.unlink(missing_ok=True)
        found_per_source[source] = found_per_source.get(source, 0) + len(accepted) - kept_before
        print(f"[{source}] new distinct images kept: {len(accepted) - kept_before}  (total {len(accepted)})")
    if len(accepted) >= TARGET_IMAGES:
        break

print(f"\nValid distinct images: {len(accepted)} / {TARGET_IMAGES}   per source: {found_per_source}")
print("Rejected/failed files:", len(failures))
if not accepted:
    raise RuntimeError(
        "No source provided usable images. Try a different OBJECT_QUERY, try again later, "
        "or collect images by hand and set OWN_IMAGES_DIR / UPLOAD_OWN_IMAGES in the settings cell."
    )

accepted_df = pd.DataFrame(accepted)
display(accepted_df[["sample_id", "image_source", "original_width", "original_height"]].head(15))

## 6. Review the RGB images — conversion has NOT started

The contact sheets below show every downloaded candidate with its sample number. Inspect them carefully.

- Enter the numbers of incorrect images in the input prompt, separated by commas (example: `3, 8, 21`).
- Leave the prompt blank if all images look correct.
- The rejected images will be removed from the working dataset.
- Then download the reviewed RGB images as a ZIP so you have a copy.

If fewer than 100 images remain, that is okay; the notebook will process the images that remain.

In [ ]:
import math
from PIL import ImageDraw
from IPython.display import display
from google.colab import files

def make_contact_sheet(records, start_index, page_size=20, columns=4):
    page_records = records[start_index:start_index + page_size]
    thumb_w, thumb_h = 220, 180
    label_h = 32
    rows = math.ceil(len(page_records) / columns)
    sheet = Image.new("RGB", (columns * thumb_w, rows * (thumb_h + label_h)), "white")
    draw = ImageDraw.Draw(sheet)

    for local_i, record in enumerate(page_records):
        global_i = start_index + local_i + 1
        with Image.open(record["image_path"]) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail((thumb_w - 12, thumb_h - 12))
            x = (local_i % columns) * thumb_w
            y = (local_i // columns) * (thumb_h + label_h)
            px = x + (thumb_w - im.width) // 2
            py = y + (thumb_h - im.height) // 2
            sheet.paste(im, (px, py))
            draw.text((x + 8, y + thumb_h + 4), f"#{global_i}  {record['sample_id']}", fill="black")
    return sheet

print(f"Reviewing {len(accepted)} downloaded images. Each image is numbered.")
for start in range(0, len(accepted), 20):
    print(f"Contact sheet: images {start + 1} to {min(start + 20, len(accepted))}")
    display(make_contact_sheet(accepted, start))

reject_text = input(
    "Enter the numbers of INCORRECT images to remove, separated by commas. "
    "Press Enter if all are correct: "
).strip()

reject_numbers = set()
if reject_text:
    for item in reject_text.split(","):
        item = item.strip()
        if item:
            number = int(item)
            if number < 1 or number > len(accepted):
                raise ValueError(f"Invalid image number: {number}")
            reject_numbers.add(number)

if reject_numbers:
    for number in sorted(reject_numbers, reverse=True):
        rejected_record = accepted.pop(number - 1)
        Path(rejected_record["image_path"]).unlink(missing_ok=True)
    # Renumber accepted samples so IDs remain consecutive.
    for i, record in enumerate(accepted, start=1):
        old_path = Path(record["image_path"])
        new_name = f"{slugify(OBJECT_QUERY)}_{i:04d}.jpg"
        new_path = IMAGES_DIR / new_name
        if old_path != new_path:
            old_path.rename(new_path)
        record["sample_id"] = f"{slugify(OBJECT_QUERY)}_{i:04d}"
        record["image_path"] = str(new_path)

if not accepted:
    raise RuntimeError("All images were rejected. Rerun the image download cell with a better query.")

for record in accepted:
    record["manual_review_status"] = "APPROVED_BY_USER_PENDING_FINAL_GATE"

print(f"Images remaining after rejection: {len(accepted)}")
display(pd.DataFrame(accepted)[["sample_id", "object_label", "manual_review_status"]].head(15))

# Package only the currently accepted RGB images for local inspection/archive.
rgb_zip_path = Path(shutil.make_archive(
    str(DATASET_DIR / f"{slugify(OBJECT_QUERY)}_reviewed_RGB_images"),
    "zip",
    root_dir=str(IMAGES_DIR),
))
print("Reviewed RGB image ZIP:", rgb_zip_path)
files.download(str(rgb_zip_path))

## 7. Approval gate — point-cloud conversion will only start after your OK

Review the images and the downloaded RGB ZIP first. If you are satisfied, type exactly `APPROVE` in the next code cell. Anything else stops the notebook before RGB2Point batch inference.

In [ ]:
approval = input("Have you checked the RGB images and approved them for point-cloud conversion? Type APPROVE: ").strip()
if approval != "APPROVE":
    raise RuntimeError("Not approved. No batch point clouds were generated. Review/reject images, then rerun this cell.")
print("Approval received. You may now run the 1,024-point conversion cell.")

## 8. Generate the original 1,024-point clouds

This runs the frozen RGB2Point checkpoint exactly as released. It writes separate `.npy` and `.ply` files with 1,024 XYZ points each.

In [ ]:
from tqdm.auto import tqdm

successful_records = []
for record in tqdm(accepted, desc="Generating original 1,024-point clouds"):
    sample_id = record["sample_id"]
    image_path = record["image_path"]
    ply_path = POINTS_1024_DIR / f"{sample_id}.ply"
    npy_path = POINTS_1024_DIR / f"{sample_id}.npy"
    try:
        points_1024 = predict_points(image_path)
        save_ply_xyz(points_1024, ply_path)
        np.save(npy_path, points_1024)
        stats = point_cloud_stats(points_1024)
        record.update({
            "generation_status": "success",
            "point_count_1024": stats["points"],
            "pointcloud_1024_ply": str(ply_path.relative_to(DATASET_DIR)),
            "pointcloud_1024_npy": str(npy_path.relative_to(DATASET_DIR)),
            "x_extent": stats["x_extent"],
            "y_extent": stats["y_extent"],
            "z_extent": stats["z_extent"],
            "min_coordinate": stats["min_coordinate"],
            "max_coordinate": stats["max_coordinate"],
            "generation_error": "",
        })
    except Exception as exc:
        record.update({
            "generation_status": "failed",
            "point_count_1024": 0,
            "pointcloud_1024_ply": "",
            "pointcloud_1024_npy": "",
            "generation_error": repr(exc),
        })
    successful_records.append(record)

manifest = pd.DataFrame(successful_records)
success_count = int((manifest["generation_status"] == "success").sum())
print(f"1,024-point clouds generated: {success_count} / {len(accepted)}")
display(manifest[["sample_id", "generation_status", "point_count_1024"]].head(15))


## 9. Download the original 1,024-point clouds

This ZIP contains only the original RGB2Point output (`.npy` and `.ply`) for successful samples.


In [ ]:
import shutil
from google.colab import files

manifest_path = DATASET_DIR / "manifest.csv"
manifest.drop(columns=["image_path", "sha256"], errors="ignore").to_csv(manifest_path, index=False)

zip_1024_path = Path(shutil.make_archive(
    str(DATASET_DIR / f"{slugify(OBJECT_QUERY)}_pointclouds_1024"),
    "zip",
    root_dir=str(POINTS_1024_DIR),
))
print("1,024-point ZIP:", zip_1024_path)
print(f"Size: {zip_1024_path.stat().st_size / (1024**2):.2f} MiB")
files.download(str(zip_1024_path))


## 10. Upsample each 1,024-point cloud to exactly 8,192 points

This is a geometric interpolation step performed after RGB2Point inference. The original 1,024-point files remain unchanged in their own folder.


In [ ]:
upsampled_records = []
for record in tqdm(successful_records, desc="Interpolating to 8,192 points"):
    if record.get("generation_status") != "success":
        continue

    sample_id = record["sample_id"]
    original_npy = POINTS_1024_DIR / f"{sample_id}.npy"
    ply_8192 = POINTS_8192_DIR / f"{sample_id}.ply"
    npy_8192 = POINTS_8192_DIR / f"{sample_id}.npy"

    points_1024 = np.load(original_npy).astype(np.float32)
    points_8192 = upsample_points_interpolation(
        points_1024,
        target_points=UPSAMPLED_POINT_COUNT,
        neighbors=5,
        seed=42,
    )
    save_ply_xyz(points_8192, ply_8192)
    np.save(npy_8192, points_8192)

    record["point_count_8192"] = int(len(points_8192))
    record["pointcloud_8192_ply"] = str(ply_8192.relative_to(DATASET_DIR))
    record["pointcloud_8192_npy"] = str(npy_8192.relative_to(DATASET_DIR))
    upsampled_records.append(record)

print(f"8,192-point clouds generated: {len(upsampled_records)}")
if upsampled_records:
    print("Example output shape:", np.load(POINTS_8192_DIR / f"{upsampled_records[0]['sample_id']}.npy").shape)

manifest = pd.DataFrame(successful_records)
manifest.drop(columns=["image_path", "sha256"], errors="ignore").to_csv(manifest_path, index=False)


## 11. Download the 8,192-point clouds

This ZIP contains the interpolated 8,192-point `.npy` and `.ply` files.


In [ ]:
zip_8192_path = Path(shutil.make_archive(
    str(DATASET_DIR / f"{slugify(OBJECT_QUERY)}_pointclouds_8192"),
    "zip",
    root_dir=str(POINTS_8192_DIR),
))
print("8,192-point ZIP:", zip_8192_path)
print(f"Size: {zip_8192_path.stat().st_size / (1024**2):.2f} MiB")
files.download(str(zip_8192_path))

## 12. Write dataset notes and create the complete dataset ZIP

In [ ]:
import shutil

readme_text = f"""RGB2Point reviewed image and point-cloud dataset
=================================================
Object query: {OBJECT_QUERY}
Search queries: {', '.join(search_queries)}
Images found per source (before your review): {found_per_source}
Reviewed RGB images retained: {len(accepted)}
Successful original 1,024-point clouds: {success_count}
Upsampled 8,192-point clouds: {len(upsampled_records)}
Model: RGB2Point, released checkpoint pc1024_three.pth
Checkpoint source: https://github.com/JaeLee18/RGB2point

Point-cloud details
-------------------
The 1,024-point files are the original frozen RGB2Point model outputs.
The 8,192-point files are post-processed by interpolation between nearby points.
Interpolation increases the number of coordinates, but does not add recovered
surface detail or make the reconstruction more accurate.
All point clouds contain XYZ only; no per-point RGB is predicted.

Research cautions
-----------------
1. Predictions for unseen object categories are out-of-domain estimates, not ground-truth scans.
2. Manually inspect point clouds before using them for training.
3. A web image search does not verify image licenses (see source_url in manifest.csv). Check permissions before training or redistribution.
4. PointLLM may require colored point clouds and point-text instruction data.
5. MetaCLIP is an image-text model; use reviewed RGB images with captions, not PLY files.

Folder layout
-------------
images/              Reviewed RGB JPEG images
pointclouds_1024/    Original RGB2Point .ply and .npy outputs
pointclouds_8192/    Interpolated 8,192-point .ply and .npy outputs
manifest.csv         Per-sample status and file paths
README.txt           This file
"""
(DATASET_DIR / "README.txt").write_text(readme_text, encoding="utf-8")

manifest.drop(columns=["image_path", "sha256"], errors="ignore").to_csv(manifest_path, index=False)

shutil.rmtree(RAW_DIR, ignore_errors=True)  # discard crawler duplicates before packaging

complete_zip_path = Path(shutil.make_archive(
    str(DATASET_DIR),
    "zip",
    root_dir=str(OUTPUT_ROOT),
    base_dir=DATASET_DIR.name,
))
print("Dataset folder:", DATASET_DIR)
print("Complete ZIP archive:", complete_zip_path)
print(f"ZIP size: {complete_zip_path.stat().st_size / (1024**2):.2f} MiB")

In [ ]:
# Optional: download the complete dataset ZIP after all stages finish.
from google.colab import files
files.download(str(complete_zip_path))

## What you get — and what still needs to happen for fine-tuning

The complete ZIP is a **reviewed RGB-image / generated-XYZ-cloud collection**, not a finished training set for both models.

- **For PointLLM:** review geometry, add/derive color information if the selected PointLLM pipeline requires colored points, and create verified point-text instruction/answer annotations. The original RGB image can help with manual annotation, but this notebook does not infer reliable point-wise colors.
- **For MetaCLIP / MetaCLIP 2:** build image-caption pairs from the original RGB images (including Bangla and English descriptions where appropriate), verify the label and locality, and fine-tune the image-text model using its supported image-text training path. The generated point clouds are not direct MetaCLIP inputs.
- **For both:** split by source/object instance where possible to reduce near-duplicate leakage; record licensing; manually reject incorrect search results and poor reconstructions; and maintain a held-out validation set.

A practical next step after smoke-testing this checkpoint is to evaluate it on a small, manually selected set of local-object images. If the shapes are not plausible, do not use the generated clouds as pseudo-ground-truth—switch to a reconstruction model trained on broader object categories or capture multi-view scans.